In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing
import math
import os

os.environ.setdefault("PYTHONHASHSEED", "0")
np.random.seed(42)

INPUT_ROOTS = ["../input", "/kaggle/input", "/kaggle/data"]

CANDIDATE_DIRS = []
for root in INPUT_ROOTS:
    CANDIDATE_DIRS.append(root)
    CANDIDATE_DIRS.append(os.path.join(root, "new-york-city-taxi-fare-prediction"))


def _find_input_file(fname: str) -> str:
    for d in CANDIDATE_DIRS:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    return os.path.join(INPUT_ROOTS[0], fname)


for r in INPUT_ROOTS:
    if os.path.exists(r):
        print(f"Listing {r}:", os.listdir(r))

train_path = _find_input_file("train.csv")
test_path = _find_input_file("test.csv")
sample_path = _find_input_file("sample_submission.csv")

print("Using paths:")
print(" train:", train_path)
print(" test :", test_path)
print(" samp :", sample_path)



Listing ../input: ['sample_submission.csv', 'test.csv.zip', 'train.csv.zip', 'labels.csv.zip', 'train.csv', 'new-york-city-taxi-fare-prediction', 'test.csv', 'labels.csv', 'description.md', 'GCP-Coupons-Instructions.rtf', 'sample_submission.csv.zip']
Listing /kaggle/input: ['sample_submission.csv', 'test.csv.zip', 'train.csv.zip', 'labels.csv.zip', 'train.csv', 'new-york-city-taxi-fare-prediction', 'test.csv', 'labels.csv', 'description.md', 'GCP-Coupons-Instructions.rtf', 'sample_submission.csv.zip']
Listing /kaggle/data: ['sample_submission.csv', 'test.csv.zip', 'train.csv.zip', 'labels.csv.zip', 'train.csv', 'new-york-city-taxi-fare-prediction', 'test.csv', 'labels.csv', 'description.md', 'GCP-Coupons-Instructions.rtf', 'sample_submission.csv.zip']
Using paths:
 train: ../input/train.csv
 test : ../input/test.csv
 samp : ../input/sample_submission.csv


In [2]:
types = {
    "fare_amount": "float32",
    "pickup_longitude": "float32",
    "pickup_latitude": "float32",
    "dropoff_longitude": "float32",
    "dropoff_latitude": "float32",
    "passenger_count": "uint8",
}

cols = [
    "key",
    "fare_amount",
    "pickup_datetime",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "passenger_count",
]

test_cols = [
    "key",
    "pickup_datetime",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "passenger_count",
]



In [3]:
train = pd.read_csv(
    train_path,
    nrows=1000000,
    usecols=cols,
    dtype={**types, **{"key": "string"}},
)
test = pd.read_csv(
    test_path,
    usecols=test_cols,
    dtype={
        **{k: v for k, v in types.items() if k != "fare_amount"},
        **{"key": "string"},
    },
)
samp = pd.read_csv(sample_path, dtype={"key": "string"})

test_id = test["key"].copy()



In [4]:
train.dropna(how="any", axis="rows", inplace=True)

if "key" in train.columns:
    before = len(train)
    train = train.drop_duplicates(subset=["key"], keep="first")
    if len(train) != before:
        print(f"Dropped {before - len(train)} duplicated training rows by key.")

mask = (
    (train["fare_amount"] > 0)
    & (train["fare_amount"] <= 250)
    & (train["passenger_count"] <= 6)
    & (train["pickup_latitude"] > -90)
    & (train["pickup_latitude"] < 90)
    & (train["dropoff_latitude"] > -90)
    & (train["dropoff_latitude"] < 90)
    & (train["pickup_longitude"] > -180)
    & (train["pickup_longitude"] < 180)
    & (train["dropoff_longitude"] > -180)
    & (train["dropoff_longitude"] < 180)
)
train = train.loc[mask]



In [5]:
nyc_geo = (
    train["pickup_longitude"].between(-74.5, -72.8)
    & train["dropoff_longitude"].between(-74.5, -72.8)
    & train["pickup_latitude"].between(40.0, 41.8)
    & train["dropoff_latitude"].between(40.0, 41.8)
)
train = train.loc[nyc_geo].copy()



In [6]:
all_data = pd.concat((train, test), ignore_index=True)

y = train["fare_amount"].to_numpy(dtype=np.float32, copy=True)
n_train = len(train)
n_test = len(test)

all_data.drop(["fare_amount", "key"], axis=1, inplace=True)




In [7]:
def week_num(day):
    """
    given the day of the month, return the week number of the month
    """
    if day <= 7:
        return "first"
    if (day > 7) and (day <= 14):
        return "second"
    if (day > 14) and (day <= 21):
        return "third"
    if (day > 21) and (day <= 28):
        return "fourth"
    return "fifth"




In [8]:
def add_time_features(data):
    data["pickup_datetime"] = pd.to_datetime(
        data["pickup_datetime"], errors="coerce", utc=True
    )

    dt = data["pickup_datetime"].dt
    data["hour"] = dt.hour
    data["day_of_week"] = dt.day_name()
    data["day_of_month"] = dt.day
    data["week_of_month"] = data["day_of_month"].map(week_num)
    data["month"] = dt.month
    data["year"] = dt.year

    data["day_of_week"] = data["day_of_week"].fillna("Unknown").astype(str)
    data["week_of_month"] = data["week_of_month"].fillna("Unknown").astype(str)

    data["hour"] = data["hour"].astype("Int64").astype(str).fillna("Unknown")
    data["month"] = data["month"].astype("Int64").astype(str).fillna("Unknown")
    data["year"] = data["year"].astype("Int64").astype(str).fillna("Unknown")

    data.drop(["day_of_month"], axis=1, inplace=True)
    return data




In [9]:
def add_geo_features(data):
    plon = data["pickup_longitude"].to_numpy(dtype=np.float32, copy=False)
    plat = data["pickup_latitude"].to_numpy(dtype=np.float32, copy=False)
    dlon = data["dropoff_longitude"].to_numpy(dtype=np.float32, copy=False)
    dlat = data["dropoff_latitude"].to_numpy(dtype=np.float32, copy=False)

    abs_diff_long = np.abs(dlon - plon)
    abs_diff_lat = np.abs(dlat - plat)

    data["abs_diff_longitude"] = abs_diff_long
    data["abs_diff_latitude"] = abs_diff_lat
    data["manhattan_distance"] = abs_diff_long + abs_diff_lat

    squared_long = abs_diff_long * abs_diff_long
    squared_lat = abs_diff_lat * abs_diff_lat
    data["squared_long"] = squared_long
    data["squared_lat"] = squared_lat
    data["euclid_distance"] = np.sqrt(squared_long + squared_lat)
    return data




In [10]:
all_data = add_time_features(all_data)



In [11]:
all_data = add_geo_features(all_data)



In [12]:
features = [
    "passenger_count",
    "hour",
    "day_of_week",
    "week_of_month",
    "month",
    "year",
    "abs_diff_longitude",
    "abs_diff_latitude",
    "manhattan_distance",
    "euclid_distance",
]

all_data = all_data[features]
all_data = pd.get_dummies(all_data)
all_data = all_data.astype(np.float32, copy=False)



In [13]:
x = all_data.iloc[:n_train]
x_test = all_data.iloc[n_train:]

X = np.ascontiguousarray(x.to_numpy(dtype=np.float32, copy=False))
X_test = np.ascontiguousarray(x_test.to_numpy(dtype=np.float32, copy=False))

train_nan_mask = ~np.isfinite(X).all(axis=1)
if train_nan_mask.any():
    keep = ~train_nan_mask
    X = X[keep]
    y = y[keep]
    n_train = X.shape[0]
    print(
        f"Dropped {train_nan_mask.sum()} training rows with NaN/inf engineered features."
    )

test_nan_mask = ~np.isfinite(X_test).all(axis=1)
if test_nan_mask.any():
    np.nan_to_num(X_test, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    print(
        f"Replaced NaN/inf in {test_nan_mask.sum()} test rows' engineered features with 0.0."
    )



In [14]:
try:
    from sklearnex import patch_sklearn

    patch_sklearn()
    print("Enabled sklearnex acceleration.")
except Exception as e:
    print("sklearnex not enabled:", repr(e))

from sklearn.ensemble import RandomForestRegressor

model_1 = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)



Enabled sklearnex acceleration.


Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [15]:
model_1.fit(X, y)

model_1_pred = model_1.predict(X_test)

model_1_pred = np.asarray(model_1_pred, dtype=np.float32)
model_1_pred = np.nan_to_num(model_1_pred, nan=0.0, posinf=0.0, neginf=0.0)
model_1_pred = np.clip(model_1_pred, 0.0, None)

sub_1 = pd.DataFrame({"key": test_id, "fare_amount": model_1_pred})
sub_1.to_csv("submission_rf.csv", index=False)



In [16]:
# BUGFIX: tensorflow.keras import crashes in this environment with
# AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'.
# Use the already-installed tf_keras package instead (same Keras API),
# keeping architecture/loss/optimizer/training loop identical.
import tf_keras as tfk

tfk.utils.set_random_seed(42)

from tf_keras.models import Sequential
from tf_keras.layers import Dense



2026-01-17 06:15:15.975203: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768630515.984288      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768630515.987049      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-17 06:15:15.998848: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [17]:
num_features = X.shape[1]



In [18]:
model = Sequential()
model.add(Dense(30, input_dim=num_features, activation="relu"))
model.add(Dense(15, activation="relu"))
model.add(Dense(7, activation="relu"))
model.add(Dense(3, activation="relu"))
model.add(Dense(1, activation="linear"))



2026-01-17 06:15:22.245816: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1768630522.246274      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:c6:00.0, compute capability: 8.6


In [19]:
model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 30)                1830      
                                                                 
 dense_1 (Dense)             (None, 15)                465       
                                                                 
 dense_2 (Dense)             (None, 7)                 112       
                                                                 
 dense_3 (Dense)             (None, 3)                 24        
                                                                 
 dense_4 (Dense)             (None, 1)                 4         
                                                                 
Total params: 2435 (9.51 KB)
Trainable params: 2435 (9.51 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [20]:
model.compile(loss="mean_squared_error", optimizer="adam")



In [21]:
model.fit(X, y, verbose=1, epochs=1)



I0000 00:00:1768630523.569441      71 service.cc:148] XLA service 0x7f604d9abc60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768630523.569481      71 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-17 06:15:23.574579: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1768630523.586212      71 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1768630523.634375      71 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


30597/30597 [==============================] - 43s 1ms/step - loss: 24.4821


In [22]:
test_pred = model.predict(X_test, verbose=0).reshape(-1)

test_pred = np.asarray(test_pred, dtype=np.float32)
test_pred = np.nan_to_num(test_pred, nan=0.0, posinf=0.0, neginf=0.0)
test_pred = np.clip(test_pred, 0.0, None)

sub = pd.DataFrame({"key": test_id, "fare_amount": test_pred})
sub.to_csv("submission_nn.csv", index=False)

print("Wrote: submission_rf.csv and submission_nn.csv")
print(sub.head())

Wrote: submission_rf.csv and submission_nn.csv
                             key  fare_amount
0    2010-10-01 21:26:11.0000001     6.762254
1   2013-10-06 01:38:00.00000083    35.738598
2    2012-03-30 19:13:53.0000001     6.303201
3    2012-02-08 02:57:23.0000001    33.012188
4  2013-12-13 22:56:00.000000237    33.991261
